# rotation-matrix-3d — ex2: compose two Rodrigues rotations and verify SO(3) closure

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `rotation-matrix-3d`. Running the final beacon cell reports progress against the `Geometry: Rotation matrix 3-D (full)` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Geometry: Rotation matrix 3-D (full)` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`rotation-matrix-3d`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "rotation-matrix-3d"
DD_SUBTOPIC = "Geometry: Rotation matrix 3-D (full)"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## 3-D rotation (Rodrigues) — deepening refresher

Rodrigues builds the rotation matrix `R = I + sinθ·K + (1-cosθ)·K²` from a unit axis `k` and angle `θ`. Composition of two rotations is MATRIX MULTIPLICATION:

```
R_total = R2 @ R1     # apply R1 first, then R2
```

**Non-commutativity.** `R1 @ R2 != R2 @ R1` in general — rotations around different axes don't commute. This is why robotics IK and camera control distinguish 'apply pitch then yaw' from 'apply yaw then pitch'.

**Closure.** The composition of two rotation matrices is ITSELF a rotation matrix. Numerical proof: `R_total @ R_total.T ≈ I` and `det(R_total) ≈ +1`. The set of 3×3 rotations forms a group (SO(3)) under matrix multiplication.

**Why this matters for ARENA.** Composing rotations is how you build a camera transform (model-space → world-space → camera-space) or chain joints in a kinematic tree.

### Exercise 2 — compose two Rodrigues rotations and verify SO(3) closure

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Analyze
> LO: Analyze the composition of two Rodrigues rotations by computing `R_total = R2 @ R1`, checking orthogonality + determinant, and exposing non-commutativity by comparing against `R1 @ R2`.
> Keywords: rotation, composition, SO(3), non-commutative
> ```

**KCs targeted:** `rotation-composition-matmul`, `verify-orthogonal-and-det-one`

A helper `rot3d(axis, theta)` is provided in the stub — it builds the Rodrigues 3-D rotation matrix exactly as in ex1.

Implement `ex2_compose_and_check(axis1, theta1, axis2, theta2)` which returns a dict with these keys:

- `'R1'`: the (3, 3) rotation from `(axis1, theta1)`.
- `'R2'`: the (3, 3) rotation from `(axis2, theta2)`.
- `'R_total'`: the composition `R2 @ R1` (apply R1 FIRST, then R2). Shape `(3, 3)`.
- `'R_swapped'`: the OTHER composition order, `R1 @ R2`. Shape `(3, 3)`.
- `'is_orthogonal'`: `True` iff `R_total @ R_total.T ≈ I` (`atol=1e-5`). Python bool.
- `'det_close_to_one'`: `True` iff `|det(R_total) - 1| < 1e-5`. Python bool.
- `'commutes'`: `True` iff `R_total ≈ R_swapped` element-wise (`atol=1e-5`). Python bool.

All four matrix values stay as `(3, 3)` `float32` tensors.

Inputs:
- `axis1`, `axis2`: `(3,)` float tensors (NOT assumed unit).
- `theta1`, `theta2`: scalar float angles in radians.

Output: dict as described above.

In [ ]:
import math

def rot3d(axis: Tensor, theta: float) -> Tensor:
    """Rodrigues 3-D rotation matrix for axis (any length) by theta radians."""
    k = axis / axis.norm()
    kx, ky, kz = k[0], k[1], k[2]
    K = t.stack([
        t.stack([t.zeros_like(kx),          -kz,                  ky]),
        t.stack([                 kz, t.zeros_like(kx),          -kx]),
        t.stack([                -ky,                kx, t.zeros_like(kx)]),
    ])
    s, c = math.sin(theta), math.cos(theta)
    return t.eye(3) + s * K + (1 - c) * (K @ K)


def ex2_compose_and_check(axis1: Tensor, theta1: float,
                          axis2: Tensor, theta2: float) -> dict:
    """Return dict with R1, R2, R_total, R_swapped, is_orthogonal, det_close_to_one, commutes."""
    raise NotImplementedError()


def _test_ex2():
    import math

    # Case 1: two different axes — non-commuting.
    axis1 = t.tensor([0.0, 0.0, 1.0])   # +z
    theta1 = math.pi / 2                # 90°
    axis2 = t.tensor([1.0, 0.0, 0.0])   # +x
    theta2 = math.pi / 3                # 60°
    out = ex2_compose_and_check(axis1, theta1, axis2, theta2)
    assert isinstance(out, dict), 'must return dict'
    for key in ['R1', 'R2', 'R_total', 'R_swapped', 'is_orthogonal', 'det_close_to_one', 'commutes']:
        assert key in out, f'missing key: {key}'

    # Shape checks.
    for key in ['R1', 'R2', 'R_total', 'R_swapped']:
        assert out[key].shape == (3, 3), f'{key} shape: {tuple(out[key].shape)}'

    # R_total = R2 @ R1.
    expected_total = out['R2'] @ out['R1']
    assert t.allclose(out['R_total'], expected_total, atol=1e-5), (
        f'R_total must equal R2 @ R1 (apply R1 first, then R2)'
    )
    # R_swapped = R1 @ R2.
    expected_swapped = out['R1'] @ out['R2']
    assert t.allclose(out['R_swapped'], expected_swapped, atol=1e-5), (
        f'R_swapped must equal R1 @ R2'
    )

    # SO(3) closure: R_total is itself orthogonal with det +1.
    assert out['is_orthogonal'] is True, 'R2 @ R1 must be orthogonal'
    assert out['det_close_to_one'] is True, 'det(R2 @ R1) must be +1'

    # Non-commutativity: rotations around DIFFERENT axes don't commute.
    assert out['commutes'] is False, (
        'z-rotation and x-rotation should NOT commute — '
        'check that you computed both R_total and R_swapped correctly'
    )

    # Case 2: same-axis rotations DO commute (R(theta_a) @ R(theta_b) = R(theta_a + theta_b)).
    axis_z = t.tensor([0.0, 0.0, 1.0])
    out2 = ex2_compose_and_check(axis_z, 0.4, axis_z, 0.7)
    assert out2['commutes'] is True, 'same-axis rotations must commute'
    assert out2['is_orthogonal'] is True
    assert out2['det_close_to_one'] is True
    # Same-axis composition equals a single rotation by the sum of angles.
    R_sum = t.eye(3)
    # Recompute via the helper exposed in the stub namespace.
    R_combined = rot3d(axis_z, 0.4 + 0.7)
    assert t.allclose(out2['R_total'], R_combined, atol=1e-5), (
        'same-axis composition should equal a single rotation by the angle sum'
    )

    # Case 3: identity composition (theta=0 for both) → R_total ≈ I.
    out3 = ex2_compose_and_check(t.tensor([1.0, 0.0, 0.0]), 0.0,
                                 t.tensor([0.0, 1.0, 0.0]), 0.0)
    assert t.allclose(out3['R_total'], t.eye(3), atol=1e-5), 'identity composition'
    assert out3['commutes'] is True, 'two identity rotations commute trivially'

    # Case 4: random non-commuting pair — SO(3) closure must hold.
    axis_a = t.tensor([1.0, 2.0, 3.0])
    axis_b = t.tensor([4.0, -1.0, 0.5])
    out4 = ex2_compose_and_check(axis_a, 0.9, axis_b, -1.3)
    assert out4['is_orthogonal'] is True
    assert out4['det_close_to_one'] is True
    # R_total @ R_total.T ≈ I (the actual numerical check).
    I = t.eye(3)
    assert t.allclose(out4['R_total'] @ out4['R_total'].T, I, atol=1e-5)
    assert abs(t.linalg.det(out4['R_total']).item() - 1.0) < 1e-5
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_compose_and_check(axis1, theta1, axis2, theta2):
    R1 = rot3d(axis1, theta1)
    R2 = rot3d(axis2, theta2)
    R_total = R2 @ R1                   # R1 first, then R2
    R_swapped = R1 @ R2                 # other order
    I = t.eye(3)
    is_orthogonal = bool(t.allclose(R_total @ R_total.T, I, atol=1e-5))
    det_close_to_one = bool(abs(t.linalg.det(R_total).item() - 1.0) < 1e-5)
    commutes = bool(t.allclose(R_total, R_swapped, atol=1e-5))
    return {
        'R1': R1,
        'R2': R2,
        'R_total': R_total,
        'R_swapped': R_swapped,
        'is_orthogonal': is_orthogonal,
        'det_close_to_one': det_close_to_one,
        'commutes': commutes,
    }
```

**Why `R2 @ R1` for 'apply R1 first'.** Conventionally a rotation acts on a column vector `v` as `R @ v`. To apply `R1` first, then `R2`, you compute `R2 @ (R1 @ v) = (R2 @ R1) @ v`. So `R_total = R2 @ R1` matches the 'first then second' reading order. (Row-vector conventions reverse this — beware when reading graphics texts.)

**SO(3) closure.** The set of 3×3 rotation matrices forms a group under matrix multiplication: closed (product of two rotations is a rotation), associative, has an identity (R(0) = I), and every element has an inverse (R(-θ) = R.T). The closure property is what `is_orthogonal` + `det_close_to_one` verify numerically.

**Why non-commutativity matters.** Robotics IK distinguishes 'pitch then yaw' from 'yaw then pitch' — they reach different poses. Camera control distinguishes 'orbit then tilt' from 'tilt then orbit'. The same axes commute (`R_z(0.4) @ R_z(0.7) == R_z(0.7) @ R_z(0.4) == R_z(1.1)`), but different axes do NOT — a fact this drill exposes via the `commutes` flag.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()